# PhonePe_ETL

## Cell-1  Import Libraries

In [1]:
from sqlalchemy import create_engine,text
import psycopg2
import os
import pandas as pd
import numpy as np
import json

print("All libraries imported successfully");

All libraries imported successfully


## Cell-2 DB Connection

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

DB_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine=create_engine(DB_URL)
pulse_path=r"C:\Users\Dell\Desktop\INTERNSHIPS\Labmentix(DS_AI_ML)\Mini Project\1\data\pulse\data"
with engine.connect() as conn:
    print("Connected to phonepe db successfully");


Connected to phonepe db successfully


## Cell-3 Aggregated_Data (3-tables)
## Aggregated_transaction

In [3]:
agg_txn_path=os.path.join(pulse_path,"aggregated","transaction","country","india","state")
agg_txn_rows=[];

for state in os.listdir(agg_txn_path):
    state_path=os.path.join(agg_txn_path,state)
    for year in os.listdir(state_path):
        year_path=os.path.join(state_path,year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter=int(file.replace(".json",""))
                file_path=os.path.join(year_path,file)
                with open(file_path,"r") as f:
                    data=json.load(f)
                txn_list=data["data"]["transactionData"]
                for txn in txn_list:
                    name=txn["name"]
                    count=txn["paymentInstruments"][0]["count"]
                    amount=txn["paymentInstruments"][0]["amount"]
                    agg_txn_rows.append(
                        {
                            "state":state,
                            "year":int(year),
                            "quarter":quarter,
                            "transaction_type":name,
                            "transaction_count":count,
                            "transaction_amount":amount
                                
                        }
                    )
df_agg_txn=pd.DataFrame(agg_txn_rows)
df_agg_txn.to_sql("aggregated_transaction",engine,if_exists="replace",index=False)
print(f"Aggregated transaction : {len(df_agg_txn)} rows loaded")
df_agg_txn.head()
                
    

Aggregated transaction : 5034 rows loaded


,state,year,quarter,transaction_type,transaction_count,transaction_amount
0,andaman-&-nicobar-islands,2018,1,Recharge & bill payments,4200,1.845307e+06
1,andaman-&-nicobar-islands,2018,1,Peer-to-peer payments,1871,1.213866e+07
2,andaman-&-nicobar-islands,2018,1,Merchant payments,298,4.525072e+05
3,andaman-&-nicobar-islands,2018,1,Financial Services,33,1.060142e+04
4,andaman-&-nicobar-islands,2018,1,Others,256,1.846899e+05


## Cell-4 Aggregated User

In [4]:
agg_user_path = os.path.join(pulse_path, "aggregated", "user", "country", "india", "state")

agg_user_rows = []

for state in os.listdir(agg_user_path):
    state_path = os.path.join(agg_user_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.replace(".json", ""))
                filepath = os.path.join(year_path, file)
                with open(filepath, "r") as f:
                    data = json.load(f)
                # usersByDevice can be None if no device data for that quarter
                users_by_device = data["data"].get("usersByDevice")
                if users_by_device:
                    for device in users_by_device:
                        agg_user_rows.append({
                            "state": state,
                            "year": int(year),
                            "quarter": quarter,
                            "brand": device["brand"],
                            "user_count": device["count"],
                            "user_percentage": device["percentage"]
                        })

df_agg_user = pd.DataFrame(agg_user_rows)
df_agg_user.to_sql("aggregated_user", engine, if_exists="replace", index=False)
print(f"aggregated_user: {len(df_agg_user)} rows loaded")
df_agg_user.head(3)

aggregated_user: 6732 rows loaded


,state,year,quarter,brand,user_count,user_percentage
0,andaman-&-nicobar-islands,2018,1,Xiaomi,1665,0.247033
1,andaman-&-nicobar-islands,2018,1,Samsung,1445,0.214392
2,andaman-&-nicobar-islands,2018,1,Vivo,982,0.145697


## Cell-5 Aggregated_insurance

In [5]:
agg_ins_path = os.path.join(pulse_path, "aggregated", "insurance", "country", "india", "state")

agg_ins_rows = []

for state in os.listdir(agg_ins_path):
    state_path = os.path.join(agg_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.replace(".json", ""))
                filepath = os.path.join(year_path, file)
                with open(filepath, "r") as f:
                    data = json.load(f)
                
                txn_list=data["data"]["transactionData"]
                for txn in txn_list:
                    name=txn["name"]
                    count=txn["paymentInstruments"][0]["count"]
                    amount=txn["paymentInstruments"][0]["amount"]
                    agg_ins_rows.append({
                        "state":state,
                        "year":int(year),
                        "quarter":quarter,
                        "insurance_type":name,
                        "insurance_count":count,
                        "insurance_amount":amount
                        
                    })

df_ins_user = pd.DataFrame(agg_ins_rows)
df_ins_user.to_sql("aggregated_insurance", engine, if_exists="replace", index=False)
print(f"aggregated_insurance: {len(df_ins_user)} rows loaded")
df_ins_user.head(3)

aggregated_insurance: 682 rows loaded


,state,year,quarter,insurance_type,insurance_count,insurance_amount
0,andaman-&-nicobar-islands,2020,2,Insurance,6,1360.0
1,andaman-&-nicobar-islands,2020,3,Insurance,41,15380.0
2,andaman-&-nicobar-islands,2020,4,Insurance,124,157975.0


## Cell-6  Map tables
## Map_transaction

In [6]:
map_txn_path=os.path.join(pulse_path,"map","transaction","hover","country","india","state")
map_txn_rows=[];

for state in os.listdir(map_txn_path):
    state_path=os.path.join(map_txn_path,state)
    for year in os.listdir(state_path):
        year_path=os.path.join(state_path,year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter=int(file.replace(".json",""))
                file_path=os.path.join(year_path,file)
                with open(file_path,"r") as f:
                    data=json.load(f)
                hover_list=data["data"]["hoverDataList"]
                for item in hover_list:
                    district=item["name"]
                    count=item["metric"][0]["count"]
                    amount=item["metric"][0]["amount"]
                    map_txn_rows.append(
                        {
                            "state":state,
                            "year":int(year),
                            "quarter":quarter,
                            "district":district,
                            "transaction_count":count,
                            "transaction_amount":amount
                                
                        }
                    )
df_map_txn=pd.DataFrame(map_txn_rows)
df_map_txn.to_sql("map_transaction",engine,if_exists="replace",index=False)
print(f"Map transaction : {len(df_map_txn)} rows loaded")
df_map_txn.head()
                
    

Map transaction : 20604 rows loaded


,state,year,quarter,district,transaction_count,transaction_amount
0,andaman-&-nicobar-islands,2018,1,north and middle andaman district,442,9.316631e+05
1,andaman-&-nicobar-islands,2018,1,south andaman district,5688,1.256025e+07
2,andaman-&-nicobar-islands,2018,1,nicobars district,528,1.139849e+06
3,andaman-&-nicobar-islands,2018,2,north and middle andaman district,825,1.317863e+06
4,andaman-&-nicobar-islands,2018,2,south andaman district,9395,2.394824e+07


## Cell-7 Map user

In [7]:
map_user_path = os.path.join(pulse_path, "map", "user", "hover","country", "india", "state")

map_user_rows = []

for state in os.listdir(map_user_path):
    state_path = os.path.join(map_user_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.replace(".json", ""))
                filepath = os.path.join(year_path, file)
                with open(filepath, "r") as f:
                    data = json.load(f)
                
                hover_data = data["data"]["hoverData"]
                for district,values in hover_data.items():
                    map_user_rows.append({
                        "state":state,
                        "year":int(year),
                        "quarter":quarter,
                        "district":district,
                        "registered_users":values["registeredUsers"],
                        "app_opens":values["appOpens"]
                    })

df_map_user = pd.DataFrame(map_user_rows)
df_map_user.to_sql("map_user", engine, if_exists="replace", index=False)
print(f"map_user: {len(df_map_user)} rows loaded")
df_map_user.head(3)

map_user: 20608 rows loaded


,state,year,quarter,district,registered_users,app_opens
0,andaman-&-nicobar-islands,2018,1,north and middle andaman district,632,0
1,andaman-&-nicobar-islands,2018,1,south andaman district,5846,0
2,andaman-&-nicobar-islands,2018,1,nicobars district,262,0


## Cell-8  Map insurance

In [8]:
map_ins_path = os.path.join(pulse_path, "map", "insurance","hover", "country", "india", "state")

map_ins_rows = []

for state in os.listdir(map_ins_path):
    state_path = os.path.join(map_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.replace(".json", ""))
                filepath = os.path.join(year_path, file)
                with open(filepath, "r") as f:
                    data = json.load(f)
                
                hover_list=data["data"]["hoverDataList"]
                for item in hover_list:
                    district=item["name"]
                    count=item["metric"][0]["count"]
                    amount=item["metric"][0]["amount"]
                    map_ins_rows.append({
                        "state":state,
                        "year":int(year),
                        "quarter":quarter,
                        "district":district,
                        "insurance_count":count,
                        "insurance_amount":amount
                        
                    })

df_ins_user = pd.DataFrame(map_ins_rows)
df_ins_user.to_sql("map_insurance", engine, if_exists="replace", index=False)
print(f"map_insurance: {len(df_ins_user)} rows loaded")
df_ins_user.head(3)

map_insurance: 13876 rows loaded


,state,year,quarter,district,insurance_count,insurance_amount
0,andaman-&-nicobar-islands,2020,2,south andaman district,3,795.0
1,andaman-&-nicobar-islands,2020,2,nicobars district,3,565.0
2,andaman-&-nicobar-islands,2020,3,north and middle andaman district,1,281.0


## Cell-9 Top tables
## Top transaction

In [9]:
top_txn_path=os.path.join(pulse_path,"top","transaction","country","india","state")
top_txn_rows=[];

for state in os.listdir(top_txn_path):
    state_path=os.path.join(top_txn_path,state)
    for year in os.listdir(state_path):
        year_path=os.path.join(state_path,year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter=int(file.replace(".json",""))
                file_path=os.path.join(year_path,file)
                with open(file_path,"r") as f:
                    data=json.load(f)
                
                for item in data["data"].get("districts",[]):
                    top_txn_rows.append(
                        {
                            "state":state,
                            "year":int(year),
                            "quarter":quarter,
                            "entity_type":"district",
                            "entity_name":item["entityName"],
                            "transaction_count":item["metric"]["count"],
                            "transaction_amount":item["metric"]["amount"]
                                
                        }
                    )
                for item in data["data"].get("pincodes",[]):
                    top_txn_rows.append({
                           "state":state,
                            "year":int(year),
                            "quarter":quarter,
                            "entity_type":"pincode",
                            "entity_name":item["entityName"],
                            "transaction_count":item["metric"]["count"],
                            "transaction_amount":item["metric"]["amount"]
                    })
df_top_txn=pd.DataFrame(top_txn_rows)
df_top_txn.to_sql("top_transaction",engine,if_exists="replace",index=False)
print(f"Top transaction : {len(df_top_txn)} rows loaded")
df_top_txn.head()
                
    

Top transaction : 18295 rows loaded


,state,year,quarter,entity_type,entity_name,transaction_count,transaction_amount
0,andaman-&-nicobar-islands,2018,1,district,south andaman,5688,1.256025e+07
1,andaman-&-nicobar-islands,2018,1,district,nicobars,528,1.139849e+06
2,andaman-&-nicobar-islands,2018,1,district,north and middle andaman,442,9.316631e+05
3,andaman-&-nicobar-islands,2018,1,pincode,744101,1622,2.769298e+06
4,andaman-&-nicobar-islands,2018,1,pincode,744103,1223,2.238042e+06


## Cell-10 Top User

In [10]:
top_user_path=os.path.join(pulse_path,"top","user","country","india","state")
top_user_rows=[];

for state in os.listdir(top_user_path):
    state_path=os.path.join(top_user_path,state)
    for year in os.listdir(state_path):
        year_path=os.path.join(state_path,year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter=int(file.replace(".json",""))
                file_path=os.path.join(year_path,file)
                with open(file_path,"r") as f:
                    data=json.load(f)
                
                for item in data["data"].get("districts",[]):
                    top_user_rows.append(
                        {
                            "state":state,
                            "year":int(year),
                            "quarter":quarter,
                            "entity_type":"district",
                            "entity_name":item["name"],
                            "registered_users":item["registeredUsers"]
                                
                        }
                    )
                for item in data["data"].get("pincodes",[]):
                    top_txn_rows.append({
                           "state":state,
                            "year":int(year),
                            "quarter":quarter,
                            "entity_type":"pincode",
                            "entity_name":item["name"],
                            "registered_users":item["registeredUsers"]
                    })
df_top_user=pd.DataFrame(top_user_rows)
df_top_user.to_sql("top_user",engine,if_exists="replace",index=False)
print(f"Top user : {len(df_top_txn)} rows loaded")
df_top_user.head()
                
    

Top user : 18295 rows loaded


,state,year,quarter,entity_type,entity_name,registered_users
0,andaman-&-nicobar-islands,2018,1,district,south andaman,5846
1,andaman-&-nicobar-islands,2018,1,district,north and middle andaman,632
2,andaman-&-nicobar-islands,2018,1,district,nicobars,262
3,andaman-&-nicobar-islands,2018,2,district,south andaman,8143
4,andaman-&-nicobar-islands,2018,2,district,north and middle andaman,911


## Cell-11 Top Insurance

In [11]:
top_ins_path = os.path.join(pulse_path, "top", "insurance", "country", "india", "state")

top_ins_rows = []

for state in os.listdir(top_ins_path):
    state_path = os.path.join(top_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.replace(".json", ""))
                filepath = os.path.join(year_path, file)
                with open(filepath, "r") as f:
                    data = json.load(f)
                # Districts
                for item in data["data"].get("districts", []):
                    top_ins_rows.append({
                        "state": state,
                        "year": int(year),
                        "quarter": quarter,
                        "entity_type": "district",
                        "entity_name": item["entityName"],
                        "insurance_count": item["metric"]["count"],
                        "insurance_amount": item["metric"]["amount"]
                    })
                # Pincodes
                for item in data["data"].get("pincodes", []):
                    top_ins_rows.append({
                        "state": state,
                        "year": int(year),
                        "quarter": quarter,
                        "entity_type": "pincode",
                        "entity_name": item["entityName"],
                        "insurance_count": item["metric"]["count"],
                        "insurance_amount": item["metric"]["amount"]
                    })

df_top_ins = pd.DataFrame(top_ins_rows)
df_top_ins.to_sql("top_insurance", engine, if_exists="replace", index=False)
print(f"top_insurance: {len(df_top_ins)} rows loaded")
df_top_ins.head(3)

top_insurance: 12276 rows loaded


,state,year,quarter,entity_type,entity_name,insurance_count,insurance_amount
0,andaman-&-nicobar-islands,2020,2,district,nicobars,3,565.0
1,andaman-&-nicobar-islands,2020,2,district,south andaman,3,795.0
2,andaman-&-nicobar-islands,2020,2,pincode,744301,3,565.0


## Cell-12 Final verification

In [12]:
tables = [
    "aggregated_transaction", "aggregated_user", "aggregated_insurance",
    "map_transaction",        "map_user",        "map_insurance",
    "top_transaction",        "top_user",        "top_insurance"
]

print("=" * 45)
print(f"{'Table':<30} {'Rows':>10}")
print("=" * 45)

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"{table:<30} {count:>10,}")

print("=" * 45)
print("ETL complete! All 9 tables loaded successfully.")

Table                                Rows
aggregated_transaction              5,034
aggregated_user                     6,732
aggregated_insurance                  682
map_transaction                    20,604
map_user                           20,608
map_insurance                      13,876
top_transaction                    18,295
top_user                            8,296
top_insurance                      12,276
ETL complete! All 9 tables loaded successfully.


top_user reloaded: 18296 rows
entity_type
pincode     10000
district     8296
Name: count, dtype: int64
